# Agent Runtime, Context & Custom Middleware Hooks -- CineBot Deep Dive

This is the companion to **`Middleware.ipynb`**, which tours the *prebuilt* middleware LangChain ships
(summarization, HITL, retries, fallback, PII, ...). This notebook goes one layer **underneath** that: the
`Runtime` / `Context` objects every hook and tool receives, the two mechanical styles for writing your
**own** middleware, and a sharper, *conditional* form of human-in-the-loop.

Concretely, you'll build and run:

1. **Setup** -- same Groq free-tier model as the companion notebook.
2. **CineBot's tools** -- a small, focused set for this notebook.
3. **Runtime & Context** -- per-call, read-only data injected into `.invoke()`, and *why the model can't
   see it by default*.
4. **`ToolRuntime`** -- the same idea, but reachable from inside a `@tool`, plus a long-term memory `store`.
5. **Node-style vs. wrap-style hooks** -- the two shapes custom middleware can take.
6. **Dynamic Prompting** -- the fix for section 3's cliffhanger: exposing context to the model on purpose.
7. **Conditional HITL** -- pausing for a human only *when a predicate says so*, not for every call.
8. **Realtime example** -- "CineBot Concierge", a multi-tenant support agent that combines all of the above.
9. **Takeaways & cheat sheet.**

> Everything here runs on **Groq's free tier** (`openai/gpt-oss-120b`) -- same model, same `.env` key as
> `Middleware.ipynb`. Read that notebook first if the CineBot tools or the `show()` helper are unfamiliar.

## Where `Runtime` / `Context` / `Store` sit in the picture

`Middleware.ipynb` covers the *agent loop* (`before_model -> MODEL -> after_model -> TOOL -> ...`). This
notebook is about the **data plumbing** that flows through that loop, which is easy to confuse:

```
create_agent(context_schema=Ctx, store=store)     <- shapes declared ONCE, when the agent is built
        |
        v
agent.invoke(input, context=Ctx(...), config={"thread_id": "..."})   <- values supplied on EVERY call
        |
        v
  Runtime[Ctx]  (handed to every hook; a ToolRuntime[Ctx] subclass is handed to every tool)
    .context            -> the Ctx instance for THIS call only -- read-only, never touched by the model
    .store              -> one shared BaseStore, the SAME object across every call and every user
    .execution_info     -> thread_id / run_id / checkpoint metadata for the call in progress
```

| Object | Scope | Mutable? | Who can read it |
|---|---|---|---|
| **context** | one `.invoke()` call | No | hooks, tools -- **never** the model directly |
| **state** (`messages`, ...) | one thread, persisted via a checkpointer | Yes | the model, hooks, tools |
| **store** | across threads *and* across users, for the life of the store | Yes | hooks, tools |

The rest of this notebook is built around that table: section 3 proves context is invisible to the model
by default, section 4 shows a tool making it visible (via a `ToolMessage`), and section 6 shows the more
direct fix (`dynamic_prompt`).

## 1. Setup

Identical pattern to `Middleware.ipynb`: Groq's free `openai/gpt-oss-120b`, `temperature=0`,
`max_retries=6` to ride out free-tier 429s. See that notebook's section 1 for the full rationale.

In [49]:
import os
import time

from dotenv import load_dotenv, find_dotenv

load_dotenv(find_dotenv())

GROQ_API_KEY = os.environ.get("GROQ_API_KEY")
assert GROQ_API_KEY, "Missing GROQ_API_KEY -- add it to the .env file at the repo root"

from langchain.chat_models import init_chat_model

GROQ_MODEL = "openai/gpt-oss-120b"


def make_model(model_id: str = GROQ_MODEL, **overrides):
    '''Build every model in one place so temperature / retry settings stay consistent.'''
    settings = {
        "temperature": 0,
        "max_retries": 6,
        "reasoning_effort": "low",   # reasoning tokens count as output -- keep them small
        **overrides,
    }
    return init_chat_model(f"groq:{model_id}", api_key=GROQ_API_KEY, **settings)


model = make_model()
print(model.invoke("Reply with exactly: CineBot Concierge is online").content)

CineBot Concierge is online


In [50]:
# --- Core LangChain ---
from dataclasses import dataclass
from typing import Callable

from langchain.agents import create_agent, AgentState
from langchain_core.tools import tool
from langchain.tools import ToolRuntime

# --- LangGraph (checkpointing, resuming, long-term memory) ---
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.store.memory import InMemoryStore
from langgraph.types import Command

# --- Middleware primitives this notebook is about ---
from langchain.agents.middleware import (
    Runtime,
    before_model,
    after_model,
    wrap_model_call,
    dynamic_prompt,
    ModelRequest,
    ModelResponse,
    ToolCallRequest,
    HumanInTheLoopMiddleware,
)


def show(result, width=260):
    '''Compact, readable dump of an agent result -- identical to the helper in Middleware.ipynb.'''
    messages = result["messages"]
    for message in messages:
        kind = type(message).__name__.replace("Message", "")
        text = message.content if isinstance(message.content, str) else str(message.content)
        text = " ".join(text.split())
        if len(text) > width:
            text = text[:width] + " ..."
        if text or not getattr(message, "tool_calls", None):
            print(f"[{kind}] {text}")
        for call in getattr(message, "tool_calls", None) or []:
            print(f"[{kind}] -> tool call: {call['name']}({call['args']})")
    if result.get("__interrupt__"):
        print("[PAUSED] waiting for a human decision (see __interrupt__)")

## 2. CineBot's tools for this notebook

A small set, each introduced right before the section that needs it:

- `check_showtimes` -- a plain, context-free tool (section 5, for the hook-timing demo).
- `fetch_preferences` -- reads `runtime.context` *and* `runtime.store` (sections 4 and 8).
- `refund_booking` -- gated by a conditional HITL `when` predicate (sections 7 and 8).

In [51]:
@tool
def check_showtimes(movie_title: str) -> str:
    '''Check available showtimes for a movie at the cinema.'''
    fake_showtimes = {
        "interstellar": "7:00 PM and 10:15 PM",
        "dune part two": "9:30 PM only",
        "oppenheimer": "Sold out for tonight",
    }
    return fake_showtimes.get(movie_title.lower(), "No showtimes found for that title.")

## 3. Runtime & Context

`context_schema` declares the **shape** of the per-call data an agent expects; the actual values are
supplied fresh on every `.invoke(..., context=...)`. Nothing about that data is static agent
configuration -- it travels with the call, the way a request-scoped dependency travels with an HTTP
request in a web framework.

Below, `log_identity` is a **node-style hook** (more on that name in section 5) that runs once before
every model call and reads `runtime.context`. It proves the value made it into the agent -- but watch
what happens when the *model itself* is asked the same question.

In [52]:
@dataclass
class CineBotContext:
    '''Per-call identity. Declared once via context_schema, supplied fresh on every .invoke().'''
    user_name: str
    is_vip: bool = False


@before_model
def log_identity(state: AgentState, runtime: Runtime[CineBotContext]) -> None:
    '''Node-style hook: runs once before every model call. Proves context is reachable in CODE.'''
    ctx = runtime.context
    print(f"[AUDIT] thread={runtime.execution_info.thread_id!r} user={ctx.user_name!r} vip={ctx.is_vip}")


context_demo_agent = create_agent(
    model=model,
    tools=[],
    context_schema=CineBotContext,
    middleware=[log_identity],
)

result = context_demo_agent.invoke(
    {"messages": [("user", "What's my name?")]},
    context=CineBotContext(user_name="Priya", is_vip=True),
    config={"configurable": {"thread_id": "context-demo-1"}},
)
show(result)

[AUDIT] thread='context-demo-1' user='Priya' vip=True
[Human] What's my name?
[AI] I don’t actually have any information about your name. If you’d like me to address you a certain way, just let me know!


**Observed:** the `[AUDIT]` line printed *before* the model ran, with `user='Priya'` -- proof the context
reached the hook. But the model's answer shows no sign of knowing the name. That's not a bug: `context` is
plumbing for your code, not an automatic addition to the prompt. Section 6 (`dynamic_prompt`) is the fix.

## 4. `ToolRuntime` -- Runtime Inside Tools

`ToolRuntime` is `Runtime`'s tool-specific subclass: same `.context` and `.store`, plus tool-only extras
(`.state`, `.config`, `.tool_call_id`). Declare a parameter typed `runtime: ToolRuntime[...]` on a `@tool`
function and LangChain injects it automatically -- **the model never sees it** in the tool's schema (it
can't fabricate a `runtime` argument, because there is no such argument to fill in).

This closes the gap from section 3: a tool's *return value* becomes a `ToolMessage`, which **does** join
the conversation the model reads. So context reached through a tool is visible to the model, even though
context read by a `before_model` hook is not.

`InMemoryStore` here plays the role of a **long-term memory** service (a real deployment would back this
with a database) -- data that outlives any single thread and is shared across every user's calls.

In [53]:
@dataclass
class CustomerContext:
    user_id: str


loyalty_store = InMemoryStore()
loyalty_store.put(("users",), "cust-501", {"preferences": "Aisle seats, no trailers, large popcorn combo"})


@tool
def fetch_preferences(runtime: ToolRuntime[CustomerContext]) -> str:
    '''Look up the current customer's saved seating and snack preferences from long-term memory.'''
    memory = runtime.store.get(("users",), runtime.context.user_id) if runtime.store else None
    return memory.value["preferences"] if memory else "No saved preferences for this customer."


pref_agent = create_agent(
    model=model,
    tools=[fetch_preferences],
    context_schema=CustomerContext,
    store=loyalty_store,
)

result = pref_agent.invoke(
    {"messages": [("user", "What are my seating and snack preferences?")]},
    context=CustomerContext(user_id="cust-501"),
)
show(result)

[Human] What are my seating and snack preferences?
[AI] -> tool call: fetch_preferences({})
[Tool] Aisle seats, no trailers, large popcorn combo
[AI] Here’s a quick reminder of what you’ve told us you like: - **Seating:** Aisle seats (you prefer easy access and a clear path out of the theater). - **Snacks:** Large popcorn combo (the big bucket of popcorn with your favorite drink). We’ll keep these preferenc ...


**Observed:** the model called `fetch_preferences` (no arguments -- everything it needed came from
`runtime`, not from the conversation), the `Tool` message carries the real preference string from the
store, and the final answer repeats it. Nobody typed `cust-501` into the chat; identity flowed entirely
through `context`.

## 5. Node-Style vs. Wrap-Style Hooks

Every prebuilt middleware in `Middleware.ipynb` is written with one of two decorator families, and you
can write your own the same way:

| Style | Decorators | Function signature | Good for |
|---|---|---|---|
| **Node-style** | `before_model`, `after_model`, `before_agent`, `after_agent` | `(state, runtime) -> dict \| Command \| None` | one-shot side effects at a fixed point: logging, gating, short-circuiting via `Command` |
| **Wrap-style** | `wrap_model_call`, `wrap_tool_call` | `(request, handler) -> response` | anything that needs *both sides* of a call in one function: timing, retry, editing the request, swapping the response |

The difference that matters in practice: a wrap-style hook calls `handler(request)` itself, so the same
function has code before **and** after the call, with the response in scope at the end. A node-style hook
only ever sees one side.

In [54]:
@before_model
def count_turns(state: AgentState, runtime: Runtime) -> None:
    '''Node-style: fires once, before the call. Has no way to see how long the call took.'''
    print(f"[TURNS] {len(state['messages'])} message(s) in state so far")


@wrap_model_call
def time_the_model(request: ModelRequest, handler: Callable[[ModelRequest], ModelResponse]) -> ModelResponse:
    '''Wrap-style: code runs on BOTH sides of handler(request), in the same function.'''
    started = time.time()
    response = handler(request)
    elapsed = time.time() - started
    print(f"[TIMING] model call took {elapsed:.2f}s")
    return response


hooked_agent = create_agent(
    model=model,
    tools=[check_showtimes],
    middleware=[count_turns, time_the_model],
)
result = hooked_agent.invoke({"messages": [("user", "What time is Dune Part Two showing?")]})
show(result)

[TURNS] 1 message(s) in state so far
[TIMING] model call took 0.53s
[TURNS] 3 message(s) in state so far
[TIMING] model call took 0.51s
[Human] What time is Dune Part Two showing?
[AI] -> tool call: check_showtimes({'movie_title': 'Dune Part Two'})
[Tool] 9:30 PM only
[AI] Dune Part Two is playing at **9:30 PM**.


**Observed:** `[TURNS]` prints once per model call (twice total here: decide-to-call-tool, then answer),
each time reporting how many messages were in state *before* that call. `[TIMING]` prints once per call
too, but only *after* `handler()` returns -- it is the only one of the two that could have retried,
edited, or discarded that response, because it is the one holding it.

This is also why `Middleware.ipynb`'s ordering rule exists: with several middleware in `middleware=[...]`,
`before_*` hooks run first-to-last and `wrap_*` hooks nest (first in the list = outermost wrapper) --
`count_turns` here always sees state *before* `time_the_model` starts timing.

## 6. Dynamic Prompting -- closing the context -> model gap

`@dynamic_prompt` is a convenience wrapper over `wrap_model_call`, specialised for one job: return a
string (or `SystemMessage`) to use as the system prompt for *this* call. Because it receives the full
`ModelRequest`, it can read `request.runtime.context` -- which is exactly how you deliberately put
per-call context in front of the model, instead of relying on a tool to smuggle it in as section 4 did.

This directly resolves section 3's cliffhanger: same `CineBotContext`, same question -- now answered.

In [55]:
@dynamic_prompt
def personalize(request: ModelRequest) -> str:
    ctx = request.runtime.context
    tier = "VIP" if ctx.is_vip else "standard"
    return (
        f"You are CineBot, a movie-ticket assistant. You are talking to {ctx.user_name}, "
        f"a {tier} member. Greet them by name when relevant and mention their tier if asked."
    )


personalized_agent = create_agent(
    model=model,
    tools=[],
    context_schema=CineBotContext,
    middleware=[personalize],
)

result = personalized_agent.invoke(
    {"messages": [("user", "What's my name, and am I a VIP?")]},
    context=CineBotContext(user_name="Priya", is_vip=True),
)
show(result)

[Human] What's my name, and am I a VIP?
[AI] Your name is **Priya**, and you’re a **VIP** member. 🎟️✨ How can I assist you with tickets today?


**Observed:** unlike section 3, the model now answers "Priya" and confirms VIP status -- the *only*
change was routing the same `runtime.context` through a system prompt instead of a print statement.
`dynamic_prompt` is the general pattern for personalization: swap `ctx.is_vip` for a locale, a
subscription tier, a feature flag, anything that should shape *how* the model behaves without touching
`tools=` or redeploying the agent.

## 7. Conditional Interrupts -- HITL that only pauses when it matters

`Middleware.ipynb` section 4 covers `HumanInTheLoopMiddleware` with a *static* `interrupt_on` -- a tool
either always pauses or never does. `InterruptOnConfig` also accepts a `when` predicate:
`Callable[[ToolCallRequest], bool]`. Now the **same tool** can auto-approve a routine call and pause only
the risky one, based on the arguments the model actually proposed.

```
tool call arrives
      |
      v
  when(request) ---- False ----> runs immediately, no human involved
      |
     True
      v
  INTERRUPT (approve / edit / reject, same as static HITL)
```

In [56]:
@tool
def refund_booking(booking_id: str, amount: float) -> str:
    '''Refund a cancelled booking.'''
    return f"Refunded ${amount:.2f} for {booking_id}."


REFUND_THRESHOLD = 100.0


def is_large_refund(request: ToolCallRequest) -> bool:
    '''The predicate: True pauses for a human, False lets the tool run immediately.'''
    amount = request.tool_call["args"].get("amount", 0)
    return amount > REFUND_THRESHOLD


conditional_agent = create_agent(
    model=model,
    tools=[refund_booking],
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "refund_booking": {
                    "allowed_decisions": ["approve", "edit", "reject"],
                    "when": is_large_refund,
                },
            },
        ),
    ],
    checkpointer=InMemorySaver(),
)

cfg_small = {"configurable": {"thread_id": "refund-small"}}
r_small = conditional_agent.invoke({"messages": [("user", "Refund booking BK1001 for $25")]}, config=cfg_small)
print("Small refund paused for approval?", bool(r_small.get("__interrupt__")))
show(r_small)

Small refund paused for approval? False
[Human] Refund booking BK1001 for $25
[AI] -> tool call: refund_booking({'amount': 25, 'booking_id': 'BK1001'})
[Tool] Refunded $25.00 for BK1001.
[AI] The booking **BK1001** has been refunded for **$25.00**. Let me know if you need anything else!


**Observed:** `$25 <= $100`, so `is_large_refund` returned `False` and the tool ran straight through --
no `[PAUSED]`, a normal final answer. Below, the same agent, same tool, but a refund over the threshold.

In [57]:
cfg_large = {"configurable": {"thread_id": "refund-large"}}
r_large = conditional_agent.invoke({"messages": [("user", "Refund booking BK1002 for $500")]}, config=cfg_large)
print("Large refund paused for approval?", bool(r_large.get("__interrupt__")))
show(r_large)

Large refund paused for approval? True
[Human] Refund booking BK1002 for $500
[AI] -> tool call: refund_booking({'amount': 500, 'booking_id': 'BK1002'})
[PAUSED] waiting for a human decision (see __interrupt__)


In [58]:
# A human approves -> the paused refund now actually runs.
resumed = conditional_agent.invoke(Command(resume={"decisions": [{"type": "approve"}]}), config=cfg_large)
show(resumed)

[Human] Refund booking BK1002 for $500
[AI] -> tool call: refund_booking({'amount': 500, 'booking_id': 'BK1002'})
[Tool] Refunded $500.00 for BK1002.
[AI] The booking **BK1002** has been successfully refunded for **$500.00**. Let me know if there’s anything else I can help you with!


In [59]:
result = conditional_agent.invoke(
    {"messages": [("user", "Refund booking BK1002 for $250")]},
    config={"configurable": {"thread_id": "refund-1"}},
)

print("big refund paused for approval?", bool(result.get("__interrupt__")))
show(result)

big refund paused for approval? True
[Human] Refund booking BK1002 for $250
[AI] -> tool call: refund_booking({'amount': 250, 'booking_id': 'BK1002'})
[PAUSED] waiting for a human decision (see __interrupt__)


In [60]:
result = conditional_agent.invoke(
    Command(resume={
        "decisions": [{
            "type": "edit",
            "edited_action": {
                "name": "refund_booking",
                "args": {"booking_id": "BK1002", "amount": 150}   # human changed 800 → 400
            }
        }]
    }),
    config={"configurable": {"thread_id": "refund-1"}},
)

In [61]:
print("After edit?", bool(result.get("__interrupt__")))
show(result)

After edit? False
[Human] Refund booking BK1002 for $250
[AI] -> tool call: refund_booking({'booking_id': 'BK1002', 'amount': 150})
[Tool] Refunded $150.00 for BK1002.
[AI] The booking **BK1002** has been refunded for **$150.00**. If you need any further assistance or a different amount, just let me know!


In [62]:
# A fresh large refund on the same thread -- this is what actually gets rejected below.
# (The previous "edit" resume already resolved the earlier interrupt, so there was nothing
# left to reject -- reject needs its OWN pending interrupt to act on.)
result = conditional_agent.invoke(
    {"messages": [("user", "Refund booking BK1004 for $600")]},
    config={"configurable": {"thread_id": "refund-1"}},
)
print("Paused for approval?", bool(result.get("__interrupt__")))
show(result)

Paused for approval? True
[Human] Refund booking BK1002 for $250
[AI] -> tool call: refund_booking({'booking_id': 'BK1002', 'amount': 150})
[Tool] Refunded $150.00 for BK1002.
[AI] The booking **BK1002** has been refunded for **$150.00**. If you need any further assistance or a different amount, just let me know!
[Human] Refund booking BK1004 for $600
[AI] -> tool call: refund_booking({'amount': 600, 'booking_id': 'BK1004'})
[PAUSED] waiting for a human decision (see __interrupt__)


In [63]:
result = conditional_agent.invoke(
    Command(resume={
        "decisions": [{
            "type": "reject",
            "message": "Refund amount too high, needs manager approval"
        }]
    }),
    config={"configurable": {"thread_id": "refund-1"}},
)
print("After rejection?", bool(result.get("__interrupt__")))
show(result)

After rejection? False
[Human] Refund booking BK1002 for $250
[AI] -> tool call: refund_booking({'booking_id': 'BK1002', 'amount': 150})
[Tool] Refunded $150.00 for BK1002.
[AI] The booking **BK1002** has been refunded for **$150.00**. If you need any further assistance or a different amount, just let me know!
[Human] Refund booking BK1004 for $600
[AI] -> tool call: refund_booking({'amount': 600, 'booking_id': 'BK1004'})
[Tool] Refund amount too high, needs manager approval
[AI] I’m unable to process that refund automatically because the amount exceeds the allowed limit. I’ll need a manager’s approval to proceed with the $600 refund for booking **BK1004**. Could you please confirm that you’d like to move forward, or let me know if the ...


**Observed:** `$500 > $100` paused the run (`[PAUSED]`, no `Tool` message yet) until the `Command(resume=
...)` approval -- at which point `refund_booking` finally executed and the model answered. Same tool,
same middleware instance, two very different experiences driven entirely by the `when` predicate.

## 8. Realtime example -- CineBot Concierge, a multi-tenant support agent

**The concept.** In production, one *compiled* agent serves many concurrent customers -- you do not
rebuild `create_agent(...)` per user. What changes on every request is the `context` you pass to
`.invoke()` and the `thread_id` in `config`. That single fact is *why* `Runtime`/`Context` exist as
something distinct from `state`:

- **`context` = per-request identity.** Like a web framework's request-scoped object (Flask's `g`,
  FastAPI's `Depends`) -- built fresh for this call, discarded after, never shared between users even
  though they hit the same compiled graph object.
- **`state` = per-conversation memory**, tied to a `thread_id`. Two different users never share a thread,
  so their `messages` never mix -- but state alone can't hold facts that should outlive *one* conversation
  (a loyalty tier, saved seating preferences).
- **`store` = cross-thread, cross-user memory**, the one object every request can reach through
  `runtime.store`, keyed by a stable id that arrives via `context` (never by something the model invents).
- **`dynamic_prompt`** turns `context` into behaviour -- personalization without redeploying the agent.
- **Conditional HITL (`when`)** keeps approval friction proportional to risk -- a five-dollar refund
  should never interrupt a human; a five-hundred-dollar one should.
- **A node-style `before_model` hook** is the cheapest way to get an audit trail (who, which thread, which
  turn) stamped on every model call, without that logic leaking into tools or prompts.

Below, **Priya** (a VIP with saved preferences) and **Rohan** (a standard member) both talk to the *same*
`concierge_agent` object, each on their own thread. Priya asks a routine question; Rohan tries a large
refund and needs approval. Watch the `[AUDIT]` lines to see the two threads never cross.

In [64]:
@dataclass
class ConciergeContext:
    user_id: str
    user_name: str
    is_vip: bool = False


# Seed long-term memory for two different customers, in the SAME store the agent will share.
loyalty_store.put(("users",), "cust-501", {"preferences": "Aisle seats, no trailers, large popcorn combo"})
loyalty_store.put(("users",), "cust-777", {"preferences": "Front row, subtitles on, no snacks"})


@tool
def fetch_customer_preferences(runtime: ToolRuntime[ConciergeContext]) -> str:
    '''Look up the current customer's saved seating and snack preferences.'''
    memory = runtime.store.get(("users",), runtime.context.user_id) if runtime.store else None
    return memory.value["preferences"] if memory else "No saved preferences for this customer."


@before_model
def audit_before_model(state: AgentState, runtime: Runtime[ConciergeContext]) -> None:
    '''Audit trail: who, which thread, which turn -- stamped on every model call, for every tenant.'''
    ctx = runtime.context
    print(
        f"[AUDIT] thread={runtime.execution_info.thread_id!r} "
        f"user={ctx.user_id}({ctx.user_name}) vip={ctx.is_vip} turn={len(state['messages'])}"
    )


@dynamic_prompt
def personalize_concierge(request: ModelRequest) -> str:
    ctx = request.runtime.context
    tier = "VIP" if ctx.is_vip else "standard"
    return (
        f"You are CineBot Concierge. You are speaking with {ctx.user_name} (customer {ctx.user_id}), "
        f"a {tier} member. Call fetch_customer_preferences before recommending seats or snacks. "
        f"When asked to process a refund, always call refund_booking with the requested amount right "
        f"away -- never explain or ask for approval yourself, the system pauses automatically for a "
        f"manager whenever that is required."
    )


concierge_agent = create_agent(
    model=model,
    tools=[fetch_customer_preferences, refund_booking],
    context_schema=ConciergeContext,
    store=loyalty_store,
    checkpointer=InMemorySaver(),
    middleware=[
        audit_before_model,
        personalize_concierge,
        HumanInTheLoopMiddleware(
            interrupt_on={
                "refund_booking": {"allowed_decisions": ["approve", "edit", "reject"], "when": is_large_refund},
            },
        ),
    ],
)
print("CineBot Concierge built:", [t.name for t in concierge_agent.get_graph().nodes if False] or "ok")

CineBot Concierge built: ok


In [65]:
# Priya: VIP, routine question, small refund -- everything should complete in one turn, no pause.
priya_cfg = {"configurable": {"thread_id": "concierge-priya"}}
priya_result = concierge_agent.invoke(
    {"messages": [("user", "What are my seat and snack preferences, and please refund booking BK9001 for $15?")]},
    context=ConciergeContext(user_id="cust-501", user_name="Priya", is_vip=True),
    config=priya_cfg,
)
show(priya_result)

[AUDIT] thread='concierge-priya' user=cust-501(Priya) vip=True turn=1
[AUDIT] thread='concierge-priya' user=cust-501(Priya) vip=True turn=3
[AUDIT] thread='concierge-priya' user=cust-501(Priya) vip=True turn=5
[Human] What are my seat and snack preferences, and please refund booking BK9001 for $15?
[AI] -> tool call: fetch_customer_preferences({})
[Tool] Aisle seats, no trailers, large popcorn combo
[AI] -> tool call: refund_booking({'amount': 15, 'booking_id': 'BK9001'})
[Tool] Refunded $15.00 for BK9001.
[AI] Here are your saved preferences: - **Seating:** Aisle seats - **Snacks:** No trailers, large popcorn combo Your refund of **$15.00** for booking **BK9001** has been processed successfully. Let me know if there’s anything else I can assist you with!


In [66]:
# Rohan: standard tier, same shared agent object, DIFFERENT thread, large refund -- should pause.
rohan_cfg = {"configurable": {"thread_id": "concierge-rohan"}}
rohan_result = concierge_agent.invoke(
    {"messages": [("user", "What are my seat and snack preferences, and please refund booking BK9002 for $300?")]},
    context=ConciergeContext(user_id="cust-777", user_name="Rohan", is_vip=False),
    config=rohan_cfg,
)
print("Rohan's refund paused for approval?", bool(rohan_result.get("__interrupt__")))
show(rohan_result)

[AUDIT] thread='concierge-rohan' user=cust-777(Rohan) vip=False turn=1
[AUDIT] thread='concierge-rohan' user=cust-777(Rohan) vip=False turn=3
Rohan's refund paused for approval? True
[Human] What are my seat and snack preferences, and please refund booking BK9002 for $300?
[AI] -> tool call: fetch_customer_preferences({})
[Tool] Front row, subtitles on, no snacks
[AI] -> tool call: refund_booking({'amount': 300, 'booking_id': 'BK9002'})
[PAUSED] waiting for a human decision (see __interrupt__)


> **`context` is per-call, not persisted.** The checkpointer saves `state` (the message history) for a
> thread, but `context` is *not* part of that saved state -- it has to be supplied again on the resume
> call, exactly like the first call. Leaving it off is a common mistake: any hook or tool that reads
> `runtime.context` on the resumed turn would otherwise see `None`.

In [67]:
# A manager reviews and approves Rohan's refund -- resumed on Rohan's own thread only.
# NOTE: context= is passed again -- it is per-call, unlike state, which the checkpointer restores for us.
rohan_resumed = concierge_agent.invoke(
    Command(resume={"decisions": [{"type": "approve"}]}),
    context=ConciergeContext(user_id="cust-777", user_name="Rohan", is_vip=False),
    config=rohan_cfg,
)
show(rohan_resumed)

[AUDIT] thread='concierge-rohan' user=cust-777(Rohan) vip=False turn=5
[Human] What are my seat and snack preferences, and please refund booking BK9002 for $300?
[AI] -> tool call: fetch_customer_preferences({})
[Tool] Front row, subtitles on, no snacks
[AI] -> tool call: refund_booking({'amount': 300, 'booking_id': 'BK9002'})
[Tool] Refunded $300.00 for BK9002.
[AI] Your seat preference is **front row** with **subtitles on**, and you don’t have any saved snack preferences. The refund of **$300** for booking **BK9002** has been processed successfully.


**Observed -- four proof points from one shared agent object:**

1. **Context isolation.** The `[AUDIT]` lines show `thread=concierge-priya` and `thread=concierge-rohan`
   with different `user_id`/`user_name`/`vip` values, even though both calls ran the exact same compiled
   `concierge_agent` -- nothing about one customer's identity leaked into the other's run.
2. **Store-backed memory.** Both customers get their *own* saved preferences back (aisle/popcorn for
   Priya, front row/subtitles for Rohan) from the one shared `loyalty_store`, keyed off `context.user_id`
   that the human never had to type.
3. **Personalization without redeploying.** `personalize_concierge` produced a different system prompt per
   call from the same middleware instance -- VIP wording for Priya, standard for Rohan.
4. **Risk-proportional approval.** Priya's $15 refund completed in a single turn; Rohan's $300 refund
   paused (`[PAUSED]`) until the explicit `Command(resume=...)` approval -- the identical `when` predicate
   from section 7, now protecting a production-shaped, multi-tenant agent instead of a toy demo.

## 9. Takeaways & cheat sheet

- **Context is not a prompt.** `context_schema` and `.invoke(context=...)` get data to your *code* (hooks,
  tools) with zero risk of the model inventing or leaking it. Getting a fact in front of the *model* is a
  deliberate extra step -- a tool result (section 4) or `dynamic_prompt` (section 6).
- **`state` vs. `store` is a lifetime question.** If a fact belongs to *this conversation*, it's `state`
  (per `thread_id`, via a checkpointer). If it should survive across conversations and across users, it
  belongs in a `store`, looked up by an id that came from `context` -- never invented by the model.
- **Node-style hooks are a single point in time; wrap-style hooks are a span.** Reach for `before_model` /
  `after_model` for one-shot logging or gating. Reach for `wrap_model_call` / `wrap_tool_call` the moment
  you need to act on *both sides* of a call in one function -- timing, retry, editing the request, or
  swapping the response (this is exactly how `ModelFallbackMiddleware` and `ToolRetryMiddleware` in
  `Middleware.ipynb` are built).
- **`when` makes HITL proportional, not binary.** A static `interrupt_on` config protects a tool
  uniformly; a `when` predicate reads the *proposed arguments* and pauses only the calls that actually
  carry risk -- the same tool can be frictionless for routine use and gated for the exceptional case.
- **One compiled agent, many tenants.** The realtime example is the pattern to copy: build
  `create_agent(...)` once; vary `context=` and `config={"thread_id": ...}` per request; keep cross-user
  facts in a `store`, not in the agent's static config.

| Need | Reach for | Key knobs |
|---|---|---|
| Per-request identity, config, DB handles | `context_schema` + `.invoke(context=...)` | a small `@dataclass`, read via `runtime.context` |
| Reading identity inside a `@tool` | `ToolRuntime[Ctx]` parameter | auto-injected, hidden from the model's tool schema |
| Cross-thread / cross-user memory | `store=` on `create_agent` + `runtime.store` | `InMemoryStore` here; swap for a real DB in production |
| One-shot side effect around the model call | `before_model` / `after_model` | `(state, runtime) -> dict \| Command \| None` |
| Acting on both sides of a call | `wrap_model_call` / `wrap_tool_call` | `(request, handler) -> response` |
| Putting context in front of the model | `dynamic_prompt` | returns a `str` / `SystemMessage` from `request.runtime.context` |
| Risk-proportional approval | `HumanInTheLoopMiddleware` + `when` | `InterruptOnConfig["when"]: Callable[[ToolCallRequest], bool]` |

### Where this shows up in production

**Multi-tenant SaaS support bots** are the realtime example almost verbatim: one deployed agent, per-request
`context` carrying tenant/user id, a shared `store` for account history, `dynamic_prompt` for brand or
locale, and `when`-gated approval on anything that touches billing.

**Internal admin tools / agentic ops:** `before_model` audit hooks are frequently a compliance requirement
on their own -- "who asked the agent to do what, and when" needs a real, structured answer, not a chat
transcript someone has to read.

**Fintech and healthcare:** conditional HITL (`when`) is how "always ask a human" gets replaced with
"ask a human only above this dollar amount / for this data category" -- friction where the risk actually
is, not everywhere uniformly.

See `Middleware.ipynb` for the full prebuilt middleware catalog -- every one of those middleware classes
is built from exactly the primitives covered here.